Set-up and create paths

In [ ]:
import geopandas as gpd
from pathlib import Path
import pandas as pd 

electricity_grouped_candidates = Path.cwd().parent / 'electricity' / 'pre_processing' / 'candidates_by_tech_group'
hydrogen_grouped_candidates = Path.cwd().parent / 'hydrogen' / 'pre_processing' / 'candidates_by_tech_group'

Combine all the electricity candidates into one layer, and do the same for all hydrogen candidates. Add a 'tech' column.

In [108]:
electricity_gdf = gpd.GeoDataFrame()
for file_path in electricity_grouped_candidates.glob('*gpkg'):
    electricity_candidates = gpd.read_file(file_path)
    electricity_candidates['tech'] = file_path.stem
    electricity_gdf = pd.concat([electricity_gdf, electricity_candidates])

In [109]:
h2_gdf = gpd.GeoDataFrame()
for file_path in hydrogen_grouped_candidates.glob('*gpkg'):
    h2_candidates = gpd.read_file(file_path)
    h2_candidates['tech'] = file_path.stem
    h2_gdf = pd.concat([h2_gdf, h2_candidates])
    

Obtain the overlapping hydrogen plants and the overlapping electricity generators

In [110]:
# Make a layer containing the overlaps between the two
overlaps = gpd.overlay(electricity_gdf, h2_gdf, 'intersection')

# Build a spatial index on overlaps
overlaps_sindex = overlaps.sindex

def overlaps_any(geom, threshold=1):
    candidates = overlaps.iloc[list(overlaps_sindex.intersection(geom.bounds))]
    return any(geom.intersection(other).area > threshold for other in candidates.geometry)

/opt/anaconda3/lib/python3.12/site-packages/geopandas/tools/overlay.py:357: UserWarning: `keep_geom_type=True` in overlay resulted in 243295 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


In [111]:
# Obtain electricity generators that overlap with hydrogen plants
electricity_overlap_indices = electricity_gdf.geometry.apply(overlaps_any)
electricity_overlaps = electricity_gdf[electricity_overlap_indices]

# Do the same for hydrogen plants that overlap with electricity generators
h2_overlap_indices = h2_gdf.geometry.apply(overlaps_any)
h2_overlaps = h2_gdf[h2_overlap_indices]

Sort the overlapping electricity generators

In [112]:
# Compute centroids
centroids = electricity_overlaps.geometry.centroid

# Sort northmost first, then west to east
electricity_overlaps_sorted = electricity_overlaps.assign(
    lon=centroids.x,
    lat=centroids.y
).sort_values(
    by=['lat', 'lon'],
    ascending=[False, True]
).reset_index(drop=True).drop(columns=['lat', 'lon'])

In [113]:
# Save the sorted position as a column
electricity_overlaps_sorted['gen_fid'] = electricity_overlaps_sorted.index

# Add a column in the h2_overlaps for the index corresponding to the overlapping electricity generator
h2_overlaps_matched = gpd.sjoin(h2_overlaps, electricity_overlaps_sorted[['geometry', 'gen_fid']], how='left', predicate='within')

In [114]:
# Output the electricity and hydrogen plant overlaps 
"""h2_overlaps_matched.to_file('h2_overlaps_aligned.gpkg', driver='GPKG')
electricity_overlaps_sorted.to_file('electricity_overlaps_sorted.gpkg', driver='GPKG')"""

"h2_overlaps_matched.to_file('h2_overlaps_aligned.gpkg', driver='GPKG')\nelectricity_overlaps_sorted.to_file('electricity_overlaps_sorted.gpkg', driver='GPKG')"

Obtain layers for the hydrogen plants and electricity generators that don't have overlaps

In [115]:
electricity_no_overlaps = electricity_gdf[~electricity_overlap_indices]
h2_no_overlaps =  h2_gdf[~h2_overlap_indices]

Resolve the overlapping sites one-by-one, saving the results to a new layer

In [116]:
rows = []

for fid, row in electricity_overlaps_sorted.iterrows():
    if fid % 2 == 1:
        rows.append(row)
    else:
        match = h2_overlaps_matched[h2_overlaps_matched['gen_fid'] == fid]
        for _, row in match.iterrows():
            rows.append(row)

overlaps_resolved = gpd.GeoDataFrame(rows, crs=electricity_overlaps_sorted.crs)

Create final layers for electricity and hydrogen plants with overlaps removed

In [117]:
resolved_h2_sites = overlaps_resolved[overlaps_resolved['dist_to_surface_flow_meters'].isna()]
resolved_h2_sites.drop(columns=['dist_to_surface_flow_meters', 'gen_fid', 'index_right'])
final_h2_sites = pd.concat([h2_no_overlaps, resolved_h2_sites])

resolved_electricity_sites = overlaps_resolved[~overlaps_resolved['dist_to_surface_flow_meters'].isna()]
resolved_electricity_sites.drop(columns=['gen_fid', 'index_right'])
final_electricity_sites = pd.concat([electricity_no_overlaps, resolved_electricity_sites])

final_h2_sites.to_file('final_h2_sites.gpkg', driver='GPKG')
final_electricity_sites.to_file('final_electricity_sites.gpkg', driver='GPKG')

Now calculate the max potentials for new electricity and hydrogen build-out by load zone and technology

In [ ]:
def calculate_potential(candidates_gdf, tech_names, ref_capacity_MW):
    """
    Calculates total potential capacity per load zone (in MW)
    for one or more technologies.
    """
    # Count number of sites per load zone
    count_by_load_area = (
        candidates_gdf.groupby("LOAD_AREA")
        .size()
        .reset_index(name="site_count")
    )

    # Add up to 3 technology name columns
    for i in range(1, 4):
        count_by_load_area[f"tech{i}"] = tech_names[i - 1] if i <= len(tech_names) else ""

    # Multiply by reference capacity (MW)
    count_by_load_area["potential_MW"] = count_by_load_area["site_count"] * ref_capacity_MW // 1

    return count_by_load_area

In [128]:
# Hydrogen technologies (tonnes/day)
ref_capacity_h2_tpd = {
    "gas_smr": 150,
    "gas_smr_ccs": 150,
    "bio_smr": 150,
    "bio_smr_ccs": 150,
    "gas_atr_ccs": 205,
    "bio_atr_ccs": 205,
    "coal_gas": 205,
    "coal_gas_ccs": 205,
    "biomass_gas": 48,
}

# Electricity technologies (MW)
ref_capacity_elec_MW = {
    "coal_igcc": 707.7,
    "coal_igcc_ccs": 707.7,
    "gas_cc": 943.5,
    "gas_cc_ccs": 943.5,
}

In [129]:

load_zones_gdf = gpd.read_file(electricity_grouped_candidates.parent / "load_zones" / "load_zones.shp")

Calculate and output hydrogen production potentials

In [ ]:
h2_groups = {
    "bio_smr_atr": ["bio_smr", "bio_smr_ccs", "bio_atr_ccs"],
    "coal_gas": ["coal_gas", "coal_gas_ccs"],
    "gas_smr_atr": ["gas_smr", "gas_smr_ccs", "gas_atr_ccs"],
    "biomass_gas": ["biomass_gas"],
}

h2_output = pd.DataFrame()

for group_name, tech_names in h2_groups.items():
    subset = final_h2_sites[final_h2_sites["tech"] == group_name]

    # Convert tonnes/day → MW using 33.39 kg H2/MWh
    ref_tpd = ref_capacity_h2_tpd[tech_names[0]]
    ref_capacity_MW = ref_tpd / 24 * 33.39

    df = calculate_potential(subset, tech_names, ref_capacity_MW)
    h2_output = pd.concat([h2_output, df], ignore_index=True)

# Expand to all load area × tech combinations
load_areas = load_zones_gdf["LOAD_AREA"].unique()
tech_sets = (
    h2_output[["tech1", "tech2", "tech3"]]
    .drop_duplicates()
    .apply(tuple, axis=1)
    .tolist()
)
all_combinations = pd.MultiIndex.from_product(
    [load_areas, tech_sets],
    names=["LOAD_AREA", "tech_set"]
).to_frame(index=False)
all_combinations[["tech1", "tech2", "tech3"]] = pd.DataFrame(
    all_combinations["tech_set"].tolist(), index=all_combinations.index
)
all_combinations = all_combinations.drop(columns="tech_set")

h2_output = all_combinations.merge(
    h2_output,
    on=["LOAD_AREA", "tech1", "tech2", "tech3"],
    how="left"
).fillna(0)

h2_output_path = "h2_tech_potentials.csv"
h2_output.to_csv(h2_output_path, index=False)
print(f"Saved hydrogen capacity by load zone to {h2_output_path}")


Saved hydrogen capacity by load zone to h2_tech_potentials.csv


Calculate and output electricity generator potentials

In [ ]:
elec_groups = {
    "coal": ["coal_igcc", "coal_igcc_ccs"],
    "gas": ["gas_cc", "gas_cc_ccs"],
}

elec_output = pd.DataFrame()

for group_name, tech_names in elec_groups.items():
    subset = final_electricity_sites[final_electricity_sites["tech"] == group_name]
    ref_capacity_MW = ref_capacity_elec_MW[tech_names[0]]

    df = calculate_potential(subset, tech_names, ref_capacity_MW)
    elec_output = pd.concat([elec_output, df], ignore_index=True)

# Expand to all load area × tech combinations
tech_sets = (
    elec_output[["tech1", "tech2", "tech3"]]
    .drop_duplicates()
    .apply(tuple, axis=1)
    .tolist()
)
all_combinations = pd.MultiIndex.from_product(
    [load_areas, tech_sets],
    names=["LOAD_AREA", "tech_set"]
).to_frame(index=False)
all_combinations[["tech1", "tech2", "tech3"]] = pd.DataFrame(
    all_combinations["tech_set"].tolist(), index=all_combinations.index
)
all_combinations = all_combinations.drop(columns="tech_set")

elec_output = all_combinations.merge(
    elec_output,
    on=["LOAD_AREA", "tech1", "tech2", "tech3"],
    how="left"
).fillna(0).drop(columns=["tech3"])

elec_output_path = "gen_tech_potentials.csv"
elec_output.to_csv(elec_output_path, index=False)
print(f"Saved generation capacity by load zone to {elec_output_path}")

Saved generation capacity by load zone to gen_tech_potentials.csv
